In [353]:
from lxml import html
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import aiohttp
import asyncio
import requests
import random
import time
import json
import re

In [354]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
})

with open("link.json") as file:
    data = json.load(file)

In [355]:
data

{'website': 'https://www.gsmarena.com/'}

In [356]:
url = data["website"]
url

'https://www.gsmarena.com/'

In [357]:
top_page = "top_page.html"

CACHE_FILE = Path("cache") / top_page

In [358]:
if CACHE_FILE.exists():
    print(f"Loading {url} from cache...")
    with open(CACHE_FILE, "r", encoding="utf-8") as file:
        html_content = file.read()
else:
    CACHE_FILE.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {url}...")
    html_content = session.get(url)
    html_content.raise_for_status()
    html_content = html_content.content

    with open(CACHE_FILE, "wb") as file:
        file.write(html_content)

Loading https://www.gsmarena.com/ from cache...


In [359]:
tree = html.fromstring(html_content)
page_brand = tree.xpath('//div[@class="brandmenu-v2 light l-box clearfix"]/ul/li/a')

In [360]:
for i in page_brand:
    brand_name = i.text
    brand_link = i.get("href")
    print(f"Link: {brand_link} \t Brand: {brand_name}")

Link: samsung-phones-9.php 	 Brand: Samsung
Link: apple-phones-48.php 	 Brand: Apple
Link: huawei-phones-58.php 	 Brand: Huawei
Link: nokia-phones-1.php 	 Brand: Nokia
Link: sony-phones-7.php 	 Brand: Sony
Link: lg-phones-20.php 	 Brand: LG
Link: htc-phones-45.php 	 Brand: HTC
Link: motorola-phones-4.php 	 Brand: Motorola
Link: lenovo-phones-73.php 	 Brand: Lenovo
Link: xiaomi-phones-80.php 	 Brand: Xiaomi
Link: google-phones-107.php 	 Brand: Google
Link: honor-phones-121.php 	 Brand: Honor
Link: oppo-phones-82.php 	 Brand: Oppo
Link: realme-phones-118.php 	 Brand: Realme
Link: oneplus-phones-95.php 	 Brand: OnePlus
Link: nothing-phones-128.php 	 Brand: Nothing
Link: vivo-phones-98.php 	 Brand: vivo
Link: meizu-phones-74.php 	 Brand: Meizu
Link: ulefone_-phones-124.php 	 Brand: Ulefone
Link: alcatel-phones-5.php 	 Brand: Alcatel
Link: zte-phones-62.php 	 Brand: ZTE
Link: rugone-phones-136.php 	 Brand: RugOne
Link: umidigi-phones-135.php 	 Brand: Umidigi
Link: coolpad-phones-105.php 	 B

In [361]:
page_brand[0].text

'Samsung'

In [368]:
brand_urls = []
new_url = url + page_brand[0].get('href')

pbar = tqdm(page_brand, ascii=True)

for i, b_url in enumerate(pbar):
    time.sleep(random.uniform(2.0, 3.0))  # Random delay between requests
    new_url = url + b_url.get('href')

    pbar.set_description(b_url.text)
    # print(f"{b_url.text} \t {b_url.get('href')}")

    html_content = session.get(new_url)
    html_content.raise_for_status()
    html_content = html_content.content

    tree = html.fromstring(html_content)
    brand_page_count = tree.xpath('//div[@class="nav-pages"]/a')

    
    paged_url = f"{url}{re.sub(r"p\d", "p{}", brand_page_count[1].get('href'))}" if len(brand_page_count) > 0 else None
    max_page = brand_page_count[-2].text if len(brand_page_count) > 0 else 0

    brand_urls.append({
        "brand": b_url.text,
        "top_url": new_url,
        "paged_url": paged_url,
        "max_page": max_page
        # "cache": str(Path("cache") / b_url.text)
    })  

TCL: 100%|##########| 36/36 [01:59<00:00,  3.32s/it]      


In [374]:
brand_page_count

[]

In [363]:
response = session.get(new_url)

print(response.status_code)
print(response.headers.get("Retry-After"))

200
None


In [369]:
metadata_dir = Path("metadata")
metadata_dir.mkdir(parents=True, exist_ok=True)

json_file = metadata_dir / "brands_urls_information.json"

with open(json_file, "w", encoding="utf-8") as f:
    json.dump(brand_urls, f, indent=4)

In [370]:
brand_urls[0]

{'brand': 'Samsung',
 'top_url': 'https://www.gsmarena.com/samsung-phones-9.php',
 'paged_url': None,
 'max_page': 0}

In [366]:
brand_urls[1]['brand']

'Apple'

In [367]:
pbar = tqdm(brand_urls, ascii=True)

for i, brand in enumerate(pbar):
    pbar.set_description(f"{brand['brand']}")

    print(f"here1 {brand['max_page']}")

    for j in range(2, brand['max_page']):  # Start from page 2 to max_page
        time.sleep(random.uniform(3.0, 4.0))  # Random delay between requests
        pbar.set_description(f"Page {j} / {brand['max_page']}")


        print("here2")

        if j == 0:
            new_url = brand['top_url']

            print("here3")
        else:

            print("here4")
            new_url = brand['paged_url']
            new_url = new_url.format(j)  # Format the URL with page number 1

        # download the page content and save it to cache
        filename = brand['brand'] + "_page" + str(j) + ".html"

        CACHE_FILE = Path("cache") / brand['brand']

        if CACHE_FILE.exists():

            print("here5")
            pbar.set_description(f"File already exists for {new_url}. Skipping download...")
            # print(f"Loading {new_url} from cache...")
            # with open(CACHE_FILE, "r", encoding="utf-8") as file:
            #     html_content = file.read()
        else:

            print("here6")
            pbar.set_description(f"Creating directory for {brand['brand']}...")
            
            # CACHE_FILE.mkdir(parents=True, exist_ok=True)

            # pbar2.set_description(f"Downloading {new_url}...")
            # html_content = requests.get(new_url)
            # html_content.raise_for_status()
            # html_content = html_content.content

            # with open(CACHE_FILE / filename, "wb") as file:
            #     file.write(html_content)


TCL: 100%|##########| 36/36 [00:00<00:00, 1540.91it/s]

here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
here1 0
